# 03 — Weather-Price Correlation

Power prices are fundamentally driven by weather:
- **Temperature** → heating/cooling demand → price
- **Wind speed** → wind generation → price suppression (ERCOT, SPP)
- **Solar radiation** → solar generation → duck curve (CAISO)

Understanding these physical drivers is critical for energy trading.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.data.fetcher import ISODataFetcher
from src.data.weather import WeatherFetcher
from src.data.processor import DataProcessor
from src.analysis.correlation import WeatherCorrelation

In [ ]:
fetcher = ISODataFetcher(config_path='../config/settings.yaml')
weather = WeatherFetcher(config_path='../config/settings.yaml')
processor = DataProcessor(config_path='../config/settings.yaml')
corr = WeatherCorrelation()

end_date = pd.Timestamp.now().strftime('%Y-%m-%d')
start_date = (pd.Timestamp.now() - pd.Timedelta(days=365)).strftime('%Y-%m-%d')

prices = fetcher.fetch_all(start_date, end_date)
weather_data = weather.fetch_all(start_date, end_date)
merged = processor.run_pipeline(prices, weather_data, save_name='exploration')
print(f"Merged dataset: {len(merged):,} rows, {len(merged.columns)} columns")

In [ ]:
# Temperature vs Price correlation by ISO
pearson = corr.pearson_by_iso(merged)
print("Temperature-Price Correlation by ISO:")
pearson.sort_values('pearson_r', ascending=False)

In [ ]:
# V-shaped temperature response
temp_response = corr.nonlinear_temp_response(merged)

fig = px.scatter(temp_response, x='temp_mid', y='mean', color='iso',
                 title='V-Shaped Temperature-Price Response',
                 labels={'temp_mid': 'Temperature (°C)', 'mean': 'Avg LMP ($/MWh)'},
                 size='count', size_max=15)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Wind and Solar impact
renewable = corr.wind_solar_impact(merged)
print("\nRenewable Energy Impact on Prices:")
renewable

In [ ]:
# CAISO Duck Curve visualization
caiso = merged[merged['iso'] == 'CAISO'].copy()
caiso['hour'] = caiso['timestamp'].dt.hour
caiso['month'] = caiso['timestamp'].dt.month

hourly = caiso.groupby(['month', 'hour']).agg(
    avg_lmp=('lmp', 'mean'),
    avg_solar=('solar_radiation', 'mean')
).reset_index()

summer = hourly[hourly['month'].isin([6, 7, 8])]
fig = go.Figure()
fig.add_trace(go.Scatter(x=summer.groupby('hour')['avg_lmp'].mean().index,
                         y=summer.groupby('hour')['avg_lmp'].mean().values,
                         name='Price', yaxis='y1'))
fig.add_trace(go.Scatter(x=summer.groupby('hour')['avg_solar'].mean().index,
                         y=summer.groupby('hour')['avg_solar'].mean().values,
                         name='Solar Radiation', yaxis='y2', line=dict(dash='dot')))
fig.update_layout(
    title='CAISO Duck Curve (Summer Months)',
    yaxis=dict(title='Avg LMP ($/MWh)'),
    yaxis2=dict(title='Solar Radiation (W/m²)', overlaying='y', side='right'),
    height=500
)
fig.show()

In [ ]:
# Lagged weather signal — does weather forecast help predict price?
lagged = corr.lagged_weather_signal(merged, max_lag=48)

fig = px.line(lagged, x='lag_hours', y='pearson_r', color='iso',
              title='Temperature-Price Correlation by Lag (hours)',
              labels={'lag_hours': 'Lag (hours)', 'pearson_r': 'Pearson r'})
fig.update_layout(height=500)
fig.show()

## Key Findings

1. **V-shaped response**: Prices increase for both extreme cold (heating) and extreme heat (cooling)
2. **CAISO duck curve**: Solar generation depresses midday prices, creating the evening ramp
3. **Wind suppression**: Negative wind-price correlation strongest in ERCOT and SPP
4. **Forecast value**: Weather signal persists across multiple lag hours — weather forecasts have trading value